# 🎥 Computer Vision Studio — Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Camillo4eyes/computer-vision-studio/blob/main/notebooks/Computer_Vision_Studio_Demo.ipynb)
[![Python](https://img.shields.io/badge/python-3.9%2B-blue)]()
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)]()

This notebook demonstrates all **11 computer vision tasks** supported by Computer Vision Studio,
running on a sample image and a short video clip.

**Tasks demonstrated:**
1. 🔍 Object Detection
2. 🎭 Instance Segmentation
3. 🧬 Semantic Segmentation
4. 🏃 Pose Estimation
5. 🏷️ Image Classification
6. 👤 Face Detection
7. 🎭 Face Mesh
8. ✋ Hand Tracking
9. 📐 Edge Detection
10. 🌊 Optical Flow
11. 🎨 Style Transfer (requires model download)


## 📦 Cell 2: Install Dependencies

In [ ]:
!pip install -q ultralytics mediapipe opencv-python-headless Pillow numpy

## 🔧 Cell 3: Imports and Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

def show(img_bgr, title='', figsize=(10, 6)):
    """Display a BGR image inline using matplotlib."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_grid(images, titles, cols=3, figsize=(15, 10)):
    """Display a grid of BGR images."""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, (img, title) in enumerate(zip(images, titles)):
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(rgb)
        axes[i].set_title(title, fontsize=11)
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

print('✅ Setup complete!')

## 🖼️ Cell 4: Download Sample Images

In [ ]:
import urllib.request
import urllib.error

def safe_download(url, dest):
    """Download a file with error handling."""
    try:
        urllib.request.urlretrieve(url, dest)
        return True
    except urllib.error.HTTPError as e:
        print(f'HTTP {e.code} error downloading {dest}: {e.reason}')
    except urllib.error.URLError as e:
        print(f'Network error downloading {dest}: {e.reason}')
    return False

# Street scene — good for object detection, segmentation, pose estimation
safe_download(
    'https://ultralytics.com/images/bus.jpg',
    'sample_image.jpg'
)

# Face image — good for face detection, mesh, hand tracking
safe_download(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/320px-Gatto_europeo4.jpg',
    'sample_cat.jpg'
)

sample = cv2.imread('sample_image.jpg')
if sample is not None:
    show(sample, '\U0001f5bc\ufe0f Sample Image')
    print(f'Image shape: {sample.shape}')
else:
    print('\u26a0\ufe0f sample_image.jpg could not be loaded.')


## 🔍 Cell 5: Object Detection (YOLOv8)

In [ ]:
from ultralytics import YOLO

model_det = YOLO('yolov8n.pt')
results = model_det('sample_image.jpg', conf=0.5, verbose=False)
annotated = results[0].plot()
show(annotated, '🔍 Object Detection — YOLOv8n')

# Summary
boxes = results[0].boxes
print(f'Detected {len(boxes)} objects')
for box in boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    print(f'  {results[0].names[cls_id]}: {conf:.2f}')

## 🎭 Cell 6: Instance Segmentation (YOLOv8-seg)

In [ ]:
model_seg = YOLO('yolov8n-seg.pt')
results_seg = model_seg('sample_image.jpg', conf=0.5, verbose=False)
seg_annotated = results_seg[0].plot()
show(seg_annotated, '🎭 Instance Segmentation — YOLOv8n-seg')
print(f'Segmented {len(results_seg[0].boxes)} instances')

## 🧬 Cell 7: Semantic Segmentation (colormap overlay)

In [ ]:
img = cv2.imread('sample_image.jpg')
results_sem = model_seg(img, conf=0.5, verbose=False)
res = results_sem[0]

h, w = img.shape[:2]
class_map = np.zeros((h, w), dtype=np.uint8)

if res.masks is not None:
    for i, mask in enumerate(res.masks.data.cpu().numpy()):
        cls_id = int(res.boxes.cls[i])
        mask_resized = cv2.resize(mask, (w, h))
        class_map[mask_resized > 0.5] = cls_id % 255

colored = cv2.applyColorMap(class_map, cv2.COLORMAP_VIRIDIS)
semantic = cv2.addWeighted(colored, 0.5, img, 0.5, 0)
show(semantic, '🧬 Semantic Segmentation — Viridis colormap')

## 🏃 Cell 8: Pose Estimation (YOLOv8-pose)

In [ ]:
model_pose = YOLO('yolov8n-pose.pt')
results_pose = model_pose('sample_image.jpg', conf=0.5, verbose=False)
pose_annotated = results_pose[0].plot()
show(pose_annotated, '🏃 Pose Estimation — YOLOv8n-pose')
if results_pose[0].keypoints is not None:
    print(f'Detected {len(results_pose[0].keypoints)} person(s)')

## 🏷️ Cell 9: Image Classification (YOLOv8-cls)

In [ ]:
model_cls = YOLO('yolov8n-cls.pt')

# Classification works better on a single object image
results_cls = model_cls('sample_cat.jpg', verbose=False)
probs = results_cls[0].probs.data.cpu().numpy()
names = results_cls[0].names
top5 = probs.argsort()[::-1][:5]

print('🏷️ Top-5 Predictions:')
for i, idx in enumerate(top5):
    print(f'  {i+1}. {names[int(idx)]}: {probs[idx]:.1%}')

## 👤 Cell 10: Face Detection (MediaPipe)

In [ ]:
import mediapipe as mp

mp_fd = mp.solutions.face_detection
img = cv2.imread('sample_image.jpg')
h, w = img.shape[:2]
annotated = img.copy()

with mp_fd.FaceDetection(model_selection=1, min_detection_confidence=0.5) as detector:
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = detector.process(rgb)
    if results.detections:
        for det in results.detections:
            bbox = det.location_data.relative_bounding_box
            x1 = int(bbox.xmin * w)
            y1 = int(bbox.ymin * h)
            bw = int(bbox.width * w)
            bh = int(bbox.height * h)
            cv2.rectangle(annotated, (x1, y1), (x1+bw, y1+bh), (0, 220, 0), 2)
        print(f'👤 Detected {len(results.detections)} face(s)')
    else:
        print('No faces detected in this sample image.')

show(annotated, '👤 Face Detection — MediaPipe')

## 🎭 Cell 11: Face Mesh (MediaPipe)

In [ ]:
mp_fm = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

img = cv2.imread('sample_image.jpg')
annotated = img.copy()

with mp_fm.FaceMesh(max_num_faces=5, refine_landmarks=True,
                    min_detection_confidence=0.5) as face_mesh:
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    if results.multi_face_landmarks:
        for face_lm in results.multi_face_landmarks:
            mp_drawing.draw_landmarks(
                annotated, face_lm,
                mp_fm.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(0,200,120), thickness=1)
            )
        print(f'🎭 Found {len(results.multi_face_landmarks)} face(s) with 468 landmarks each')
    else:
        print('No faces detected.')

show(annotated, '🎭 Face Mesh — MediaPipe (468 landmarks)')

## ✋ Cell 12: Hand Tracking (MediaPipe)

In [ ]:
mp_hands = mp.solutions.hands

# Download a hand image for this demo
safe_download(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c6/Hand_sign_for_the_number_1.jpg/320px-Hand_sign_for_the_number_1.jpg',
    'sample_hand.jpg'
)

img = cv2.imread('sample_hand.jpg')
if img is None:
    print('\u26a0\ufe0f Hand image not available. Skipping hand tracking demo.')
else:
    annotated = img.copy()

    with mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5) as hands:
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)
        if results.multi_hand_landmarks:
            for hand_lm in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, hand_lm,
                    mp_hands.HAND_CONNECTIONS,
                    mp_styles.get_default_hand_landmarks_style(),
                    mp_styles.get_default_hand_connections_style()
                )
            print(f'\u270b Detected {len(results.multi_hand_landmarks)} hand(s)')
        else:
            print('No hands detected.')

    show(annotated, '\u270b Hand Tracking \u2014 MediaPipe Hands')


## 📐 Cell 13: Edge Detection (Canny / Sobel / Laplacian)

In [ ]:
img = cv2.imread('sample_image.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Canny
canny = cv2.Canny(gray, 50, 150)
canny_bgr = cv2.cvtColor(canny, cv2.COLOR_GRAY2BGR)

# Sobel
sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel = np.clip(cv2.magnitude(sx, sy), 0, 255).astype(np.uint8)
sobel_bgr = cv2.cvtColor(sobel, cv2.COLOR_GRAY2BGR)

# Laplacian
lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
laplacian = np.clip(np.abs(lap), 0, 255).astype(np.uint8)
lap_bgr = cv2.cvtColor(laplacian, cv2.COLOR_GRAY2BGR)

show_grid(
    [img, canny_bgr, sobel_bgr, lap_bgr],
    ['Original', 'Canny', 'Sobel', 'Laplacian'],
    cols=2, figsize=(14, 8)
)

## 🌊 Cell 14: Optical Flow (Farneback)

In [ ]:
# Simulate two consecutive frames by slightly translating the image
img = cv2.imread('sample_image.jpg')
h, w = img.shape[:2]

M1 = np.float32([[1, 0, 0], [0, 1, 0]])
M2 = np.float32([[1, 0, 5], [0, 1, 3]])  # shift 5px right, 3px down

frame1 = cv2.warpAffine(img, M1, (w, h))
frame2 = cv2.warpAffine(img, M2, (w, h))

gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

flow = cv2.calcOpticalFlowFarneback(
    gray1, gray2, None,
    pyr_scale=0.5, levels=3, winsize=15,
    iterations=3, poly_n=5, poly_sigma=1.2, flags=0
)

mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
hsv = np.zeros_like(img)
hsv[..., 0] = ang * 180 / np.pi / 2
hsv[..., 1] = 255
hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

show_grid(
    [frame1, frame2, flow_rgb],
    ['Frame 1', 'Frame 2 (shifted)', '🌊 Optical Flow (HSV)'],
    cols=3, figsize=(15, 5)
)

## 🎨 Cell 15: Neural Style Transfer

> **Note:** This cell downloads a ~6 MB pre-trained model file from Stanford.
> It may take a moment depending on your connection.

In [ ]:
import os
import urllib.error

model_url = 'https://cs.stanford.edu/people/jcjohns/fast-neural-style/models/instance_norm/mosaic.t7'
model_path = 'mosaic.t7'

if not os.path.exists(model_path):
    downloaded = False
    # Try HTTPS first, then HTTP as fallback if the server rejects the secure request
    urls_to_try = [model_url, model_url.replace('https://', 'http://', 1)]
    for attempt_url in urls_to_try:
        try:
            urllib.request.urlretrieve(attempt_url, model_path)
            print('Model downloaded.')
            downloaded = True
            break
        except urllib.error.HTTPError as e:
            print(f'HTTP {e.code} error for {attempt_url}: {e.reason}')
        except urllib.error.URLError as e:
            print(f'URL error for {attempt_url}: {e.reason}')
    if not downloaded:
        print('\u26a0\ufe0f Could not download style model. The Stanford model server may be unavailable.')
        print('Please try again later or download the model manually from:')
        print(model_url)

if os.path.exists(model_path):
    img = cv2.imread('sample_image.jpg')
    h, w = img.shape[:2]

    net = cv2.dnn.readNetFromTorch(model_path)
    blob = cv2.dnn.blobFromImage(
        img, 1.0, (512, int(h * 512 / w)),
        (103.939, 116.779, 123.680), swapRB=False, crop=False
    )
    net.setInput(blob)
    output = net.forward()

    output = output.squeeze().transpose(1, 2, 0)
    output += np.array([103.939, 116.779, 123.680])
    styled = np.clip(output, 0, 255).astype(np.uint8)
    styled = cv2.resize(styled, (w, h))

    show_grid([img, styled], ['Original', '\U0001f3a8 Mosaic Style Transfer'], cols=2, figsize=(14, 6))
else:
    print('\u23ed\ufe0f Skipping style transfer demo (model not available).')


## 🔀 Cell 16: Combined Mode — Multi-Task Grid

Apply multiple tasks to the same frame and display results as a grid.

In [ ]:
img = cv2.imread('sample_image.jpg')

# ── Task 1: Object Detection
res_det = model_det(img, conf=0.5, verbose=False)
det_annotated = res_det[0].plot()

# ── Task 2: Instance Segmentation
res_seg2 = model_seg(img, conf=0.5, verbose=False)
seg_annotated2 = res_seg2[0].plot()

# ── Task 3: Pose Estimation
res_pose2 = model_pose(img, conf=0.5, verbose=False)
pose_annotated2 = res_pose2[0].plot()

# ── Task 4: Edge Detection (Canny)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 50, 150)
edge_annotated = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

show_grid(
    [det_annotated, seg_annotated2, pose_annotated2, edge_annotated],
    ['🔍 Object Detection', '🎭 Instance Segmentation',
     '🏃 Pose Estimation', '📐 Edge Detection (Canny)'],
    cols=2, figsize=(14, 10)
)

## 🎬 Cell 17: Video Demo — Process a Few Frames

Download a short sample video, extract a few frames, run object detection,
and save results as a GIF.

In [ ]:
from IPython.display import Image as IPyImage
import imageio

# Download a short sample video
video_available = safe_download(
    'https://ultralytics.com/assets/decelera_landscape_min.mov',
    'sample_video.mov'
)

if not video_available:
    print('\u23ed\ufe0f Skipping video demo (sample video could not be downloaded).')
else:
    cap = cv2.VideoCapture('sample_video.mov')
    frames_out = []
    frame_count = 0
    max_frames = 20

    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % 2 == 0:  # sample every other frame
            res = model_det(frame, conf=0.5, verbose=False)
            annotated = res[0].plot()
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            frames_out.append(annotated_rgb)
        frame_count += 1

    cap.release()

    if frames_out:
        imageio.mimsave('detection_demo.gif', frames_out, fps=5)
        print(f'Saved {len(frames_out)}-frame GIF')
        IPyImage('detection_demo.gif')
    else:
        print('No frames processed.')


## 📷 Cell 18: Webcam Demo (Google Colab)

Capture a frame from the Colab webcam and run inference.

> **Note:** This cell only works in Google Colab with a webcam attached.
> It will be skipped automatically outside of Colab.

In [ ]:
try:
    from google.colab.patches import cv2_imshow
    from google.colab.output import eval_js
    from base64 import b64decode
    from IPython.display import Javascript

    js_code = Javascript('''
    async function captureFrame() {
        const video = document.createElement('video');
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        video.srcObject = stream;
        await video.play();
        await new Promise(r => setTimeout(r, 2000)); // warm up
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getTracks().forEach(t => t.stop());
        return canvas.toDataURL('image/jpeg', 0.8);
    }
    captureFrame().then(result => element.textContent = result);
    ''')
    display(js_code)
    data_url = eval_js('captureFrame()')
    _, img_bytes = data_url.split(',', 1)
    img_data = b64decode(img_bytes)

    import io
    nparr = np.frombuffer(img_data, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    res = model_det(frame, conf=0.5, verbose=False)
    annotated = res[0].plot()
    cv2_imshow(annotated)
    print('✅ Webcam capture and inference complete!')

except Exception as e:
    print(f'⚠️ Webcam demo skipped (not in Colab or webcam unavailable): {e}')